# Local FinQA LoRA pilot

A small feasibility experiment: pinned SmolLM2-135M-Instruct, 32 train examples,
8 seeded held-out dev examples, rank-8 LoRA on q/v projections, 384-token sequences,
and at most 24 optimizer steps. The original 1.5B experiment is unchanged.

**Target: under 30 minutes with cached dependencies/model/data.** The runner has a
25-minute cooperative budget and a 10-minute training cap. Initial downloads,
installation, and a single operation can exceed this budget. Start on AC power.

This is a small-model pilot, not evidence of Qwen/QLoRA performance. Question-ranked evidence can still omit important context; tiny-model answer accuracy can be zero. Both models
receive identical budgeted prompts and answer-only targets. Report the actual
sample count, training steps, and wall time. Loss reduction alone does not establish
financial reasoning quality or a speedup over full fine-tuning.


In [ ]:
from pathlib import Path
import os
import sys

ROOT = Path.cwd()
if not (ROOT / "configs").exists():
    ROOT = ROOT.parent
assert (ROOT / "src" / "llm_finetuning").exists(), "Open this notebook inside the repository"
os.chdir(ROOT)
sys.path.insert(0, str(ROOT / "src"))
os.environ.setdefault("HF_HOME", str(ROOT / "data" / "huggingface"))
os.environ.setdefault("HF_HUB_DISABLE_XET", "1")


Select the repository `.venv` as the notebook kernel. If dependencies are missing,
run the following cell once (network access required), then restart the kernel.
It installs only pilot requirements, avoiding the full TRL/MLflow/serving stack.
The CPU build of PyTorch works; CUDA is optional. This does not replace PyTorch.


In [ ]:
# Uncomment only if required; installation is outside the experiment timing.
# %pip install "torch>=2.4,<3" "transformers>=5.19,<6" "peft>=0.14,<1" "datasets>=3,<5" "accelerate>=1,<2" "pydantic>=2.9,<3" "PyYAML>=6,<7"
import torch
print("PyTorch:", torch.__version__)
print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")


## Run the experiment

The first run downloads about 270 MB of model weights plus tokenizer and FinQA
files. Subsequent runs reuse verified source files and the model cache. Each run
writes a new folder, so previous results remain available. Incomplete runs retain
completed-stage metrics and an explicit error in `metrics.json`.


In [ ]:
from llm_finetuning.pilot import run_pilot

output_dir, result = run_pilot("configs/local-pilot.yaml", budget_minutes=25)


## Results for presentation

`completion_loss` is token-weighted cross-entropy on held-out answer tokens only;
perplexity is exp(loss). Protocol v2 requires a complete numeric/percent answer (relative tolerance 0.0001)
and scores yes/no and text answers by normalized exact match. Numeric accuracy
uses numeric/percent references only; answer accuracy uses all examples. These
are custom pilot metrics, not official FinQA execution scoring. At 8 examples, one answer changes overall
accuracy by 12.5 percentage points; the numeric denominator may be smaller. Evidence is ranked using question terms only, within a fixed prompt budget;
question, instructions and chat framing remain intact. Generation allows up to
64 tokens and honors model/tokenizer EOS tokens. Evidence omission, EOS stops,
token-limit stops and parse failures are recorded in error_analysis.json. These
conditions do not prove why an answer is wrong. Evaluation time includes loss and generation;
it is not a steady-state inference benchmark. Predictions are saved for inspection.


In [ ]:
import csv
from IPython.display import Markdown, display

rows = list(csv.DictReader((output_dir / "comparison.csv").open(encoding="utf-8")))
def percentage(value):
    return f"{float(value):.1%}" if value else "N/A"

display(Markdown(
    "| Model | Answer loss | Numeric accuracy | Numeric n | Overall accuracy | Dev n | Eval seconds |\n"
    "|---|---:|---:|---:|---:|---:|---:|\n" + "\n".join(
        f"| {r['model']} | {float(r['completion_loss']):.3f} | "
        f"{percentage(r['numeric_accuracy'])} | {r['numeric_examples']} | "
        f"{percentage(r['answer_accuracy'])} | {r['examples']} | "
        f"{float(r['evaluation_seconds']):.1f} |" for r in rows
    )
))
print("Status:", result["status"])
print("Optimizer steps:", result["training"]["steps"])
print("Trainable parameters:", result["parameters"])
print("Total minutes:", round(result["total_seconds"] / 60, 2))
print("Artifacts:", output_dir.resolve())
for name in ("baseline", "adapted"):
    print(name, "answer accuracy:", result[name]["answer_accuracy"])
    print("Answer types:", result[name]["answer_type_metrics"])
    print("Diagnostics:", result[name]["error_analysis"])


In [ ]:
for before, after in zip(result["baseline"]["predictions"], result["adapted"]["predictions"]):
    print("ID:", before["id"], "Reference:", before["reference"])
    print("Baseline:", before["prediction"])
    print("Adapted:", after["prediction"])
    print()


Suggested slide caption: **Exploratory CPU/4-GB-GPU feasibility pilot: 135M model,
32 training examples, 8 held-out dev examples, one seed, answer-only LoRA tuning.**
Protocol v2 changes context length, evidence selection, generation cap and scoring;
do not compare its metrics directly with the historical v1 README result.
Copy measured values from the table; include any regressions, zero accuracy, time
caps, and generation truncations. Export `comparison.csv` and `metrics.json` with
the slides. The adapter and resolved configuration are saved beside them.

Model: https://huggingface.co/HuggingFaceTB/SmolLM2-135M-Instruct

Dataset: Chen et al., *FinQA: A Dataset of Numerical Reasoning over Financial Data*,
EMNLP 2021; original pinned source revision and SHA-256 hashes are in `metrics.json`.
